# WaterVision AI — provenance, licenses, and artifact identity

**Dataset credit:** Rudy, I. M., & Wilson, M. J. (2024). *Underwater Image/Turbidity Dataset* (1.0.0) [Data set]. Zenodo. https://doi.org/10.5281/zenodo.10951021. The dataset record declares AGPL-3.0-or-later. The related paper is separately CC BY 4.0: Rudy & Wilson (2024), *PeerJ* 12:e18254, https://doi.org/10.7717/peerj.18254. Do not transfer the paper license to dataset images. Do not redistribute raw images from this notebook without carrying upstream notices and checking any image-level restrictions.
The notebook selects 662 images with usable FNU labels from the upstream Turbidivision v1.0.0 release. Project bands: low <5 FNU; moderate 5–<20 FNU; high ≥20 FNU. Prepared images are RGB, EXIF-oriented, resized within 1024×1024, and saved JPEG quality 92.
**Model artifact:** `yolo26n_full_v1` is the Experiment 01 full-run model, distinct from Experiment 02 CV fold checkpoints. Its training configuration in the source notebook is pretrained `yolo26n-cls.pt`, up to 50 epochs, imgsz 320, batch 16, SGD lr0 0.005 with cosine schedule, weight decay 0.0005, patience 10, seed 42, deterministic=True, AMP=False, auto_augment=None, erasing=0.0.
The ONNX file in the companion repository is the owner-supplied export for this full run. Experiment 02 pooled CV metrics apply only to its three separate fold models, not this ONNX artifact.
**Licensing:** This project uses Ultralytics YOLO and follows AGPL-3.0 for the project; the source dataset declares AGPL-3.0-or-later. Ultralytics states trained models are AGPL-3.0 by default; an Enterprise license is the alternative described by Ultralytics. This repository does not claim an Enterprise license and does not claim MIT. Exporting to ONNX does not erase upstream terms. See `LICENSE`, `docs/licensing.md`, and `docs/dataset.md`.

# WaterVision AI: YOLO26n-cls turbidity classification
This is a workflow scaffold, not the original research notebook. Verify data rights and family-aware splits before use. Outputs are not potability or field-safety assessments.

In [ ]:
%pip install -q ultralytics scikit-learn pandas matplotlib


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/WaterVisionAI')
DATA_DIR = Path('/content/watervision_data')  # set to authorized train/<class>, val/<class> folders
print('Project folder:', PROJECT_DIR, 'Dataset ready:', DATA_DIR.exists())

## Confirm prepared splits
This validates directory structure only; it does not validate family-level separation. Use the original family manifest for CV.

In [ ]:
for split in ('train', 'val'):
    path = DATA_DIR / split
    assert path.is_dir(), f'Missing {path}'
    print(split, [p.name for p in path.iterdir() if p.is_dir()])

## Train a demonstration model
This generic run is not a reproduction of the reported CV. Train each family-aware fold separately to reproduce that experiment, retaining distinct checkpoints.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo26n-cls.pt')
model.train(data=str(DATA_DIR), epochs=50, imgsz=224, batch=32, device=0, name='watervision_demo')

## Inference with a specific checkpoint
Choose a checkpoint and record its fold identity. Do not combine or relabel fold checkpoints as a unified model.

In [ ]:
from pathlib import Path
from ultralytics import YOLO
WEIGHTS = Path('/content/runs/classify/watervision_demo/weights/best.pt')
IMAGE = Path('/content/example.jpg')  # change to an authorized image
assert WEIGHTS.is_file() and IMAGE.is_file()
result = YOLO(str(WEIGHTS)).predict(source=str(IMAGE), imgsz=224)[0]
for i in result.probs.data.argsort(descending=True).tolist():
    print(result.names[i], float(result.probs.data[i]))